# Competition Analysis - AI Actors Network

Ce notebook produit une carte concurrentielle des entreprises IA a partir de `database.db`.
oi
## Objectif

Cartographier les proximites concurrentielles a partir des relations `entreprise -> concurrent`, puis projeter cet espace en 2D.

## Pipeline

1. Extraire les entreprises avec concurrents et score financier.
2. Nettoyer et normaliser les noms de concurrents.
3. Construire les couples diriges `entreprise -> concurrent`.
4. Agreger et construire la matrice dirigee `entreprise x concurrents`.
5. Projeter en 2D (t-SNE) et generer une carte interactive.

## Convention de lecture

Chaque section suit le schema: **Objectif -> Entrees -> Traitement -> Sorties**.

## Regles metier

- Population analysee: `ranking_score > 0`.
- `ranking_score = capitalisation`, ou `fonds leves` si capitalisation absente.
- Taille des labels sur la carte: mode retenu `sqrt` (plage compacte 8-18).

## 1. Configuration et imports

**Objectif**: charger les dependances et definir les constantes communes.

**Entrees**: chemin du projet.
**Sorties**: variables globales (`DB_PATH`, `EXPORTS_DIR`, noms de colonnes, seed).

In [1]:
import sqlite3
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

ROOT        = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DB_PATH     = ROOT / "database.db"
EXPORTS_DIR = ROOT / "analyses" / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

COL_NAME        = "name"
COL_SECTOR      = "sector"
COL_COMPETITORS = "main_competitors"
RANDOM_SEED     = 42

print(f"Base : {DB_PATH}")
print(f"Exports : {EXPORTS_DIR}")

Base : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\database.db
Exports : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports


## 2. Extraction SQL des entreprises et du score financier

**Objectif**: produire le tableau brut des entreprises avec concurrents.

**Entrees**: table `enterprises` (`name`, `sector`, `main_competitors`, `capitalization`, `funds_raised`).
**Traitement**: parsing robuste des montants, puis calcul de `ranking_score = COALESCE(capitalization, funds_raised)`.
**Sorties**: `df_raw` trie par `ranking_score` desc et filtre sur score positif.

In [27]:
SQL = f"""
WITH base AS (
    SELECT
        name             AS {COL_NAME},
        sector           AS {COL_SECTOR},
        main_competitors AS {COL_COMPETITORS},

        -- Nettoie les formats numeriques heterogenes (espaces, NBSP, virgules/points).
        CASE
            WHEN NULLIF(TRIM(capitalization), '') IS NULL THEN NULL
            ELSE CASE
                WHEN (
                    LENGTH(REPLACE(REPLACE(REPLACE(TRIM(capitalization), ' ', ''), CHAR(160), ''), ',', '.'))
                    - LENGTH(REPLACE(REPLACE(REPLACE(REPLACE(TRIM(capitalization), ' ', ''), CHAR(160), ''), ',', '.'), '.', ''))
                ) > 1
                THEN CAST(REPLACE(REPLACE(REPLACE(REPLACE(TRIM(capitalization), ' ', ''), CHAR(160), ''), ',', '.'), '.', '') AS REAL)
                ELSE CAST(REPLACE(REPLACE(REPLACE(TRIM(capitalization), ' ', ''), CHAR(160), ''), ',', '.') AS REAL)
            END
        END AS cap_musd,

        CASE
            WHEN NULLIF(TRIM(funds_raised), '') IS NULL THEN NULL
            ELSE CASE
                WHEN (
                    LENGTH(REPLACE(REPLACE(REPLACE(TRIM(funds_raised), ' ', ''), CHAR(160), ''), ',', '.'))
                    - LENGTH(REPLACE(REPLACE(REPLACE(REPLACE(TRIM(funds_raised), ' ', ''), CHAR(160), ''), ',', '.'), '.', ''))
                ) > 1
                THEN CAST(REPLACE(REPLACE(REPLACE(REPLACE(TRIM(funds_raised), ' ', ''), CHAR(160), ''), ',', '.'), '.', '') AS REAL)
                ELSE CAST(REPLACE(REPLACE(REPLACE(TRIM(funds_raised), ' ', ''), CHAR(160), ''), ',', '.') AS REAL)
            END
        END AS funds_musd

    FROM enterprises
    WHERE main_competitors IS NOT NULL
      AND main_competitors != ''
)
SELECT
    {COL_NAME},
    {COL_SECTOR},
    {COL_COMPETITORS},
    COALESCE(cap_musd, funds_musd) AS ranking_score
FROM base
WHERE COALESCE(cap_musd, funds_musd) IS NOT NULL
  AND COALESCE(cap_musd, funds_musd) > 0
ORDER BY ranking_score DESC
"""

with sqlite3.connect(DB_PATH) as con:
    df_raw = pd.read_sql_query(SQL, con)

print(
    f"{len(df_raw)} entreprises retenues "
    f"(sélection: capitalisation, sinon fonds levés)"
)
df_raw[["name", "ranking_score"]].head(10)

95 entreprises retenues (sélection: capitalisation, sinon fonds levés)


,name,ranking_score
0,Nvidia,5000000.0
1,Google,4560000.0
2,Alphabet Inc.,4120000.0
3,Apple,4000000.0
4,Microsoft,3450000.0
5,Amazon,2900000.0
6,Broadcom,1835000.0
7,Meta,1503000.0
8,SpaceX,1480000.0
9,AWS,1200000.0


In [28]:
# Export brut
raw_path = EXPORTS_DIR / "competitors_raw.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Export brut → {raw_path}")

Export brut → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_raw.csv


## 3. Nettoyage syntaxique des concurrents

**Objectif**: transformer la liste texte des concurrents en liste propre exploitable.

**Traitement**: split multi-separateurs, suppression des valeurs invalides, normalisation de casse, suppression des auto-references.
**Sortie**: `df_clean` avec une liste de concurrents par entreprise.

In [29]:
_SEP = re.compile(r"[,;/\n]+")
_INVALID = re.compile(r"^(na|n/a|none|unknown|tbd|-)$", re.IGNORECASE)

def clean_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\(.*?\)", "", s).strip()
    s = re.sub(r"\s{2,}", " ", s)
    return s.title()

def split_competitors(raw: str) -> list[str]:
    parts = _SEP.split(str(raw))
    return [clean_name(p) for p in parts
            if clean_name(p) and not _INVALID.match(p.strip())]

df_clean = df_raw.copy()
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(split_competitors)

df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS] if c.lower() != r[COL_NAME].lower()],
    axis=1,
)

df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

print(f"{len(df_clean)} entreprises après nettoyage")
df_clean[[COL_NAME, COL_COMPETITORS]].head(8)

95 entreprises après nettoyage


,name,main_competitors
0,Nvidia,"[Amd, Intel, Google, Broadcom, Qualcomm]"
1,Google,"[Microsoft, Openai, Perpexity Ai, Amazon, Byte..."
2,Alphabet Inc.,"[Microsoft, Amazon, Apple, Meta, Oracle, Opena..."
3,Apple,"[Google, Microsoft, Samsung, Amazon, Xiaomi, H..."
4,Microsoft,"[Amazon, Google, Meta, Apple, Sony, Oracle]"
5,Amazon,"[Walmart, Temu, Shein, Alibaba, Microsoft, Goo..."
6,Broadcom,"[Nvidia, Qualcomm, Intel, Marvell Technology]"
7,Meta,"[Google, Bytedance, Snap, X, Apple, Snapchat, ..."


## 3b. Normalisation semantique des noms

**Objectif**: regrouper filiales, marques et variantes sous un nom canonique.

**Entrees**: `df_clean` et dictionnaire `SEMANTIC_ALIASES`.
**Sortie**: `df_clean` semantiquement normalise, dedoublonne, sans auto-concurrence.

In [30]:
SEMANTIC_ALIASES: dict[str, str] = {
    # ── Google (nom canonique dans la base) ───────────────────────────────────
    "Alphabet":             "Google",
    "Alphabet Inc.":        "Google",
    "Youtube":              "Google",
    "YouTube":              "Google",
    "Deepmind":             "Google",
    "DeepMind":             "Google",
    "Google Deepmind":      "Google",
    "Google DeepMind":      "Google",
    "Google Brain":         "Google",
    "Google Cloud":         "Google",
    "Google Cloud Platform":"Google",
    "GCP":                  "Google",
    "Waymo":                "Google",
    "Verily":               "Google",
    "Calico":               "Google",
    "Waze":                 "Google",
    "Google Translate":     "Google",
    "Gmail":                "Google",
    "Android":              "Google",
    # ── Meta (nom canonique dans la base) ─────────────────────────────────────
    "Meta Platforms":       "Meta",
    "Facebook":             "Meta",
    "Instagram":            "Meta",
    "Whatsapp":             "Meta",
    "WhatsApp":             "Meta",
    "Threads":              "Meta",
    "Oculus":               "Meta",
    "Meta Quest":           "Meta",
    "LLaMA":                "Meta",
    "Llama":                "Meta",
    # ── Microsoft ─────────────────────────────────────────────────────────────
    "Azure":                "Microsoft",
    "Microsoft Azure":      "Microsoft",
    "Linkedin":             "Microsoft",
    "LinkedIn":             "Microsoft",
    "Github":               "Microsoft",
    "GitHub":               "Microsoft",
    "Skype":                "Microsoft",
    "Bing":                 "Microsoft",
    "Nuance":               "Microsoft",
    "Nuance Communications":"Microsoft",
    "Activision Blizzard":  "Microsoft",
    "Activision":           "Microsoft",
    "Xbox":                 "Microsoft",
    "Microsoft Translator": "Microsoft",
    "Office 365":           "Microsoft",
    # ── Amazon ────────────────────────────────────────────────────────────────
    "Aws":                  "AWS",
    "AWS":                  "AWS",
    "Alexa":                "Amazon",
    "Twitch":               "Amazon",
    "Amazon.Com":           "Amazon",
    "Amazon Prime":         "Amazon",
    "Amazon Prime Video":   "Amazon",
    "Kindle":               "Amazon",
    # ── Amazon Web Services (entité séparée dans la base) ─────────────────────
    "Amazon Web Services":  "AWS",
    "Amazon Web Services (AWS)": "AWS",
    # ── Apple ─────────────────────────────────────────────────────────────────
    "Siri":                 "Apple",
    "Apple Inc.":           "Apple",
    "Apple Inc":            "Apple",
    "Iphone":               "Apple",
    "iPhone":               "Apple",
    "Ipad":                 "Apple",
    "iPad":                 "Apple",
    "Apple Silicon":        "Apple",
    # ── Salesforce ────────────────────────────────────────────────────────────
    "Slack":                "Salesforce",
    "Tableau":              "Salesforce",
    "Mulesoft":             "Salesforce",
    "MuleSoft":             "Salesforce",
    "Salesforce - Einstein":"Salesforce",
    "Einstein":             "Salesforce",
    # ── IBM ───────────────────────────────────────────────────────────────────
    "Red Hat":              "IBM",
    "RedHat":               "IBM",
    "Watsonx":              "IBM",
    "Watson":               "IBM",
    "IBM Watson":           "IBM",
    # ── Oracle ────────────────────────────────────────────────────────────────
    "Netsuite":             "Oracle",
    "NetSuite":             "Oracle",
    "Java":                 "Oracle",
    # ── Nvidia (nom canonique dans la base) ───────────────────────────────────
    "NVIDIA":               "Nvidia",
    "Cuda":                 "Nvidia",
    "CUDA":                 "Nvidia",
    "Nvidia Corporation":   "Nvidia",
    # ── INtel (nom avec cette casse dans la base) ─────────────────────────────
    "INtel":                "Intel",
    "Intel":                "Intel",
    "Intel Corporation":    "Intel",
    # ── AMD ───────────────────────────────────────────────────────────────────
    "Amd":                  "AMD",
    "AMD Inc.":             "AMD",
    "Advanced Micro Devices": "AMD",
    # ── ByteDance ─────────────────────────────────────────────────────────────
    "Tiktok":               "ByteDance",
    "TikTok":               "ByteDance",
    "Douyin":               "ByteDance",
    "Bytedance":            "ByteDance",
    "Bytedanse":            "ByteDance",
    # ── X (nom canonique dans la base, anciennement Twitter) ──────────────────
    "Twitter":              "X",
    "X.Com":                "X",
    "X Corp":               "X",
    # ── Tesla ─────────────────────────────────────────────────────────────────
    "Tesla Inc.":           "Tesla",
    "Tesla Motors":         "Tesla",
    # ── SpaceX ────────────────────────────────────────────────────────────────
    "Space Exploration Technologies": "SpaceX",
    "Starlink":             "SpaceX",
    # ── Baidu ─────────────────────────────────────────────────────────────────
    "Ernie":                "Baidu",
    "Ernie Bot":            "Baidu",
    "ERNIE":                "Baidu",
    # ── Tencent ───────────────────────────────────────────────────────────────
    "Wechat":               "Tencent",
    "WeChat":               "Tencent",
    "Qq":                   "Tencent",
    "QQ":                   "Tencent",
    # ── OpenAI ────────────────────────────────────────────────────────────────
    "Chatgpt":              "OpenAI",
    "ChatGPT":              "OpenAI",
    "Gpt-4":                "OpenAI",
    "GPT-4":                "OpenAI",
    "Gpt4":                 "OpenAI",
    "GPT4":                 "OpenAI",
    "Openai":               "OpenAI",
    "Perpexity Ai":         "Perplexity AI",
    "DALL-E":               "OpenAI",
    "Dall-E":               "OpenAI",
    "Sora":                 "OpenAI",
    # ── Anthropic ─────────────────────────────────────────────────────────────
    "Claude":               "Anthropic",
    "Claude AI":            "Anthropic",
    # ── Samsung ───────────────────────────────────────────────────────────────
    "Samsung Electronics":  "Samsung",
    "Samsung System LSI":   "Samsung",
    # ── Adobe ─────────────────────────────────────────────────────────────────
    "Adobe Firefly":        "Adobe",
    "Photoshop":            "Adobe",
    # ── Qualcomm ──────────────────────────────────────────────────────────────
    "Qualcomm Inc.":        "Qualcomm",
    # ── SAP ───────────────────────────────────────────────────────────────────
    "SAP SE":               "SAP",
    # ── Huawei ────────────────────────────────────────────────────────────────
    "Huawei Technologies":  "Huawei",
    # ── Alibaba ───────────────────────────────────────────────────────────────
    "Alibaba Group":        "Alibaba",
    "AliCloud":             "Alibaba",
    "Alipay":               "Alibaba",
    # ── HP ────────────────────────────────────────────────────────────────────
    "Hewlett-Packard":      "HP",
    "Hewlett Packard":      "HP",
    # ── Netflix ───────────────────────────────────────────────────────────────
    "Netflix Inc.":         "Netflix",
    # ── Spotify ───────────────────────────────────────────────────────────────
    "Spotify AB":           "Spotify",
    # ── Sony ──────────────────────────────────────────────────────────────────
    "Sony Corporation":     "Sony",
    "PlayStation":          "Sony",
    # ── Dell ──────────────────────────────────────────────────────────────────
    "Dell Technologies":    "Dell",
    # ── Lenovo ────────────────────────────────────────────────────────────────
    "Lenovo Group":         "Lenovo",
    # ── MediaTek ──────────────────────────────────────────────────────────────
    "MediaTek Inc.":        "MediaTek",
    # ── Xiaomi ────────────────────────────────────────────────────────────────
    "Xiaomi Corporation":   "Xiaomi",
    # ── Oppo ──────────────────────────────────────────────────────────────────
    "OPPO":                 "Oppo",
    # ── Vivo ──────────────────────────────────────────────────────────────────
    "VIVO":                 "Vivo",
    # ── Disney ────────────────────────────────────────────────────────────────
    "Disney+":              "Disney",
    "Walt Disney":          "Disney",
    # ── Boeing ────────────────────────────────────────────────────────────────
    "Boeing Company":       "Boeing",
    # ── Autres acteurs spécifiques IA ─────────────────────────────────────────
    "Mistral":              "Mistral AI",
    "Stability AI":         "Stability",
    "Stability.ai":         "Stability",
    "Midjourney Inc.":      "Midjourney",
    "Runway ML":            "Runway",
}

def apply_semantic_aliases(names: list[str]) -> list[str]:
    return [SEMANTIC_ALIASES.get(n, n) for n in names]

df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(apply_semantic_aliases)
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(lambda lst: list(dict.fromkeys(lst)))
df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS]
               if c.lower() != r[COL_NAME].lower()
               and SEMANTIC_ALIASES.get(r[COL_NAME].title(), r[COL_NAME]).lower() != c.lower()],
    axis=1,
)
df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

# explode garde le nom COL_COMPETITORS, pas "competitor"
preview = df_clean[[COL_NAME, COL_COMPETITORS]].explode(COL_COMPETITORS)
print(f"{len(SEMANTIC_ALIASES)} aliases | {len(df_clean)} entreprises après normalisation sémantique")
print("\nTop concurrents après normalisation :")
print(preview[COL_COMPETITORS].value_counts().head(15).to_string())

148 aliases | 95 entreprises après normalisation sémantique

Top concurrents après normalisation :
main_competitors
Google       27
Microsoft    22
OpenAI       17
Meta         13
Amazon       11
Anthropic    11
ByteDance     9
Apple         9
Nvidia        7
Intel         6
Qualcomm      6
Tencent       6
Tesla         5
Oracle        5
AMD           4


## 4. Format long dirigé `entreprise -> concurrent`

**Objectif**: passer du format liste au format relationnel ligne a ligne.

**Traitement**: explode, canonicalisation des noms via la base, dedoublonnage des paires dirigees.
**Sortie**: `df_long` et export `competitors_long.csv`.

In [31]:
def _norm_key(value: str) -> str:
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

# Référence de canonicalisation: tous les noms d'entreprises de la base.
with sqlite3.connect(DB_PATH) as con:
    df_all_names = pd.read_sql_query("SELECT name FROM enterprises WHERE name IS NOT NULL", con)

canonical_by_key = {}
for name in df_all_names["name"].tolist():
    clean = str(name).strip()
    if clean:
        key = _norm_key(clean)
        if key not in canonical_by_key:
            canonical_by_key[key] = clean

df_long = (
    df_clean
    .explode(COL_COMPETITORS)
    .rename(columns={COL_COMPETITORS: "competitor"})
    .reset_index(drop=True)
    [[COL_NAME, "competitor", COL_SECTOR, "ranking_score"]]
)

# Canonicalise les compétiteurs pour éviter les doublons de casse/écriture.
df_long["competitor"] = (
    df_long["competitor"]
    .map(lambda c: canonical_by_key.get(_norm_key(c), c))
)

# Force quelques formes canoniques stables, même si la base contient des variantes historiques.
preferred_canonical = {
    "INtel": "Intel",
    "Amd": "AMD",
    "Bytedanse": "ByteDance",
    "Perpexity Ai": "Perplexity AI",
    "Amazon Web Services": "AWS",
}
df_long["competitor"] = df_long["competitor"].map(lambda c: preferred_canonical.get(c, c))

# Dédoublonne au niveau entreprise→compétiteur (dataset dirigé).
df_long = (
    df_long.drop_duplicates(subset=[COL_NAME, "competitor"])
    .reset_index(drop=True)
)
df_long.insert(0, "pair_id", df_long.index)

long_path = EXPORTS_DIR / "competitors_long.csv"
df_long.to_csv(long_path, index=False)

print(f"{len(df_long)} couples entreprise→concurrent après normalisation et dédoublonnage")
print(f"Entreprises (lignes): {df_long[COL_NAME].nunique()} | Compétiteurs (colonnes potentielles): {df_long['competitor'].nunique()}")
print(f"Export → {long_path}")
df_long.head(10)

571 couples entreprise→concurrent après normalisation et dédoublonnage
Entreprises (lignes): 95 | Compétiteurs (colonnes potentielles): 365
Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_long.csv


,pair_id,name,competitor,sector,ranking_score
0,0,Nvidia,AMD,"Hardware, AI model, ICT",5000000.0
1,1,Nvidia,Intel,"Hardware, AI model, ICT",5000000.0
2,2,Nvidia,Google,"Hardware, AI model, ICT",5000000.0
3,3,Nvidia,Broadcom,"Hardware, AI model, ICT",5000000.0
4,4,Nvidia,Qualcomm,"Hardware, AI model, ICT",5000000.0
5,5,Google,Microsoft,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
6,6,Google,OpenAI,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
7,7,Google,Perplexity AI,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
8,8,Google,Amazon,"AI model, Media & Entertainment, Sales & Marke...",4560000.0
9,9,Google,ByteDance,"AI model, Media & Entertainment, Sales & Marke...",4560000.0


## 5. Agregation des relations

**Objectif**: compter la frequence de chaque paire `entreprise -> concurrent`.

**Sortie**: `df_agg` (colonnes `name`, `competitor`, `count`) et export `competitors_aggregated.csv`.

In [32]:
df_agg = (
    df_long
    .groupby([COL_NAME, "competitor"], sort=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 paires entreprise–concurrent :")
display(df_agg.head(20))

print("\nDistribution des fréquences :")
display(df_agg["count"].describe())

agg_path = EXPORTS_DIR / "competitors_aggregated.csv"
df_agg.to_csv(agg_path, index=False)
print(f"\nExport → {agg_path}")

Top 20 paires entreprise–concurrent :


,name,competitor,count
0,Nvidia,AMD,1
1,Nvidia,Intel,1
2,Nvidia,Google,1
3,Nvidia,Broadcom,1
4,Nvidia,Qualcomm,1
5,Google,Microsoft,1
6,Google,OpenAI,1
7,Google,Perplexity AI,1
8,Google,Amazon,1
9,Google,ByteDance,1



Distribution des fréquences :


count    571.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: count, dtype: float64


Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_aggregated.csv


## 6. Matrice dirigee `entreprise x concurrents`

**Objectif**: preparer la representation matricielle pour la projection.

**Lignes**: entreprises retenues.
**Colonnes**: concurrents mentionnes.
**Sortie**: `df_matrix` et export `cooccurrence_matrix.csv` (matrice dirigee).

In [33]:
# Matrice dirigée: lignes = entreprises, colonnes = compétiteurs.
# On garde la logique asymétrique, sans M + M^T.
df_matrix = df_agg.pivot_table(
    index=COL_NAME,
    columns="competitor",
    values="count",
    fill_value=0,
)

enterprise_actors = sorted(df_raw[COL_NAME].dropna().astype(str).str.strip().unique())
df_matrix = df_matrix.reindex(index=enterprise_actors, fill_value=0)

print(f"Matrice dirigée: {df_matrix.shape[0]} entreprises × {df_matrix.shape[1]} compétiteurs")
print(f"Densité non-nulle : {(df_matrix.values > 0).mean():.1%}")

cooc_path = EXPORTS_DIR / "cooccurrence_matrix.csv"
df_matrix.to_csv(cooc_path)
print(f"Export matrice dirigée → {cooc_path}")

Matrice dirigée: 95 entreprises × 365 compétiteurs
Densité non-nulle : 1.6%
Export matrice dirigée → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\cooccurrence_matrix.csv


## 7. Projection 2D (t-SNE)

**Objectif**: projeter les profils de concurrence des entreprises en 2D.

**Entree**: `df_matrix` normalisee en L2.
**Parametre cle**: `perplexity = 10`.
**Sortie**: `df_coords` et export `coords_2d.csv`.

In [34]:
N = df_matrix.shape[0]
X = normalize(df_matrix.values, norm="l2")

perplexity = 10
print(f"t-SNE 2D sur {N} entreprises (perplexity={perplexity})")
coords = TSNE(
    n_components=2,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    metric="cosine",
    random_state=RANDOM_SEED,
).fit_transform(X)
method = "t-SNE"

out_degree = df_matrix.sum(axis=1).to_dict()
ranking_score_map = df_raw.set_index(COL_NAME)["ranking_score"].to_dict()

enterprise_names = list(df_matrix.index)
df_coords = pd.DataFrame({"actor": enterprise_names, "x": coords[:, 0], "y": coords[:, 1]})
df_coords["score"] = df_coords["actor"].map(out_degree).fillna(0)
df_coords["log_score"] = np.log10(df_coords["score"] + 1)
df_coords["ranking_score"] = df_coords["actor"].map(ranking_score_map).fillna(0)

sector_map = (
    df_raw.set_index(COL_NAME)[COL_SECTOR]
    .dropna()
    .apply(lambda s: s.split(",")[0].strip())
    .to_dict()
)
df_coords["sector"] = df_coords["actor"].map(sector_map).fillna("Unknown")

coords_path = EXPORTS_DIR / "coords_2d.csv"
df_coords.to_csv(coords_path, index=False)
print(f"Coordonnées 2D ({method}) → {coords_path}")
df_coords.sort_values("ranking_score", ascending=False).head(10)

t-SNE 2D sur 95 entreprises (perplexity=10)
Coordonnées 2D (t-SNE) → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\coords_2d.csv


,actor,x,y,score,log_score,ranking_score,sector
62,Nvidia,43.105267,42.139858,5.0,0.778151,5000000.0,Hardware
47,Google,-19.091406,39.222549,8.0,0.954243,4560000.0,AI model
12,Alphabet Inc.,-13.530030,41.904823,10.0,1.041393,4120000.0,Cloud Provider
19,Apple,-1.761398,-5.786357,16.0,1.230449,4000000.0,ICT
58,Microsoft,-5.058219,37.987854,6.0,0.845098,3450000.0,Cloud Provider
14,Amazon,0.119929,3.103106,10.0,1.041393,2900000.0,Unknown
25,Broadcom,53.911198,54.154305,4.0,0.698970,1835000.0,Hardware
57,Meta,-1.750517,45.148956,12.0,1.113943,1503000.0,AI model
83,SpaceX,-21.402464,-16.256607,8.0,0.954243,1480000.0,Cloud Provider
7,AWS,-2.307393,12.368585,5.0,0.778151,1200000.0,Hardware


## 8. Carte interactive et exports finaux

**Objectif**: generer la visualisation finale.

**Regles visuelles**:
- Couleur par pays (contraintes de palette appliquees).
- Taille du label basee sur `ranking_score` en mode `sqrt` (plage 8-18).
- Affichage labels uniquement, sans points.

**Sortie**: `competition_map_2d.html` + recapitulatif des fichiers exportes.

In [35]:
import plotly.graph_objects as go

info_cols = [
    "founded_year", "employees_count", "revenue_millions",
    "capitalization", "funds_raised", "description"
]

# Strategie de taille des labels (solution recommandee)
SIZE_MODE = "sqrt"  # options: linear, linear_clip_p99, log10, log10_clip_p99, log1p, log1p_clip_p99, sqrt, sqrt_clip_p99
SIZE_MIN, SIZE_MAX = 8, 18
SIZE_FALLBACK = 10
SIZE_CLIP_Q = 0.99


def country_key(value: str) -> str:
    return str(value).strip().casefold()


def scale_to_range(values: pd.Series, min_size: float, max_size: float, fallback_size: float) -> pd.Series:
    if values.isna().all() or np.isclose(values.max(), values.min()):
        return pd.Series(fallback_size, index=values.index)
    scaled = min_size + (values - values.min()) * (max_size - min_size) / (values.max() - values.min())
    return scaled


def compute_label_sizes(
    raw_scores: pd.Series,
    mode: str,
    min_size: float,
    max_size: float,
    fallback_size: float,
    clip_q: float,
) -> tuple[pd.Series, dict]:
    scores = pd.to_numeric(raw_scores, errors="coerce").where(lambda s: s > 0)
    if not scores.notna().any():
        return pd.Series(fallback_size, index=raw_scores.index), {"mode": mode, "clip_value": None}

    use_clip = "_clip_p99" in mode
    base_mode = mode.replace("_clip_p99", "")

    working = scores.copy()
    clip_value = None
    if use_clip:
        clip_value = float(working.quantile(clip_q))
        working = working.clip(upper=clip_value)

    if base_mode == "linear":
        transformed = working
    elif base_mode == "log10":
        transformed = np.log10(working)
    elif base_mode == "log1p":
        transformed = np.log1p(working)
    elif base_mode == "sqrt":
        transformed = np.sqrt(working)
    else:
        raise ValueError(f"SIZE_MODE inconnu: {mode}")

    sizes = scale_to_range(transformed, min_size=min_size, max_size=max_size, fallback_size=fallback_size).round(1)
    return sizes, {"mode": mode, "clip_value": clip_value}


def apply_label_repel(
    df: pd.DataFrame,
    x_col: str = "x",
    y_col: str = "y",
    size_col: str = "label_size",
    iterations: int = 320,
    anchor_pull: float = 0.02,
    step: float = 0.55,
) -> pd.DataFrame:
    """
    Petit moteur de repulsion (type force layout) pour limiter les chevauchements de labels.
    La simulation tourne en coordonnees normalisees [0,1] puis revient en coordonnees d'origine.
    """
    x = pd.to_numeric(df[x_col], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(df[y_col], errors="coerce").to_numpy(dtype=float)
    s = pd.to_numeric(df[size_col], errors="coerce").fillna(SIZE_FALLBACK).to_numpy(dtype=float)

    n = len(df)
    if n <= 1:
        return pd.DataFrame({"x_label": x, "y_label": y}, index=df.index)

    xmin, xmax = float(np.min(x)), float(np.max(x))
    ymin, ymax = float(np.min(y)), float(np.max(y))
    xspan = xmax - xmin if not np.isclose(xmax, xmin) else 1.0
    yspan = ymax - ymin if not np.isclose(ymax, ymin) else 1.0

    pos = np.column_stack([(x - xmin) / xspan, (y - ymin) / yspan])
    origin = pos.copy()

    smin, smax = float(np.min(s)), float(np.max(s))
    sspan = smax - smin if not np.isclose(smax, smin) else 1.0
    radii = 0.010 + ((s - smin) / sspan) * 0.028

    for _ in range(iterations):
        disp = np.zeros_like(pos)

        for i in range(n - 1):
            delta = pos[i] - pos[i + 1 :]
            dist = np.linalg.norm(delta, axis=1) + 1e-9
            target = radii[i] + radii[i + 1 :]
            overlap = target - dist
            mask = overlap > 0

            if np.any(mask):
                force = (overlap[mask] / dist[mask])[:, None] * delta[mask]
                move = force * step
                disp[i] += move.sum(axis=0)
                disp[i + 1 :][mask] -= move

        disp += (origin - pos) * anchor_pull
        pos += disp
        pos = np.clip(pos, 0.0, 1.0)

    x_label = pos[:, 0] * xspan + xmin
    y_label = pos[:, 1] * yspan + ymin
    return pd.DataFrame({"x_label": x_label, "y_label": y_label}, index=df.index)


def build_country_colors(countries: list[str]) -> dict[str, str]:
    forced_colors = {
        "china": "#D62828",          # rouge
        "united states": "#1D4ED8",  # bleu
        "japan": "#6B7280",          # gris
        "south korea": "#9CA3AF",    # gris
        "korea, south": "#9CA3AF",   # gris
    }

    europe_country_keys = {
        "albania", "andorra", "austria", "belarus", "belgium", "bosnia and herzegovina",
        "bulgaria", "croatia", "cyprus", "czechia", "czech republic", "denmark", "estonia",
        "finland", "france", "germany", "greece", "hungary", "iceland", "ireland", "italy",
        "latvia", "liechtenstein", "lithuania", "luxembourg", "malta", "moldova", "monaco",
        "montenegro", "netherlands", "north macedonia", "norway", "poland", "portugal", "romania",
        "san marino", "serbia", "slovakia", "slovenia", "spain", "sweden", "switzerland",
        "ukraine", "united kingdom", "vatican city", "kosovo"
    }

    greens = [
        "#1B4332", "#24553F", "#2D6A4F", "#3A7D5D", "#4C956C", "#5FAF7D",
        "#74C69D", "#52B788", "#40916C", "#3E8E63", "#2F7D57", "#2A6F54"
    ]

    fallback_palette = [
        "#6D597A", "#E76F51", "#264653", "#457B9D", "#BC6C25", "#B56576",
        "#3A86FF", "#FF006E", "#0A9396", "#7F5539", "#4361EE", "#FF7F11",
        "#2B2D42", "#8D99AE", "#8338EC", "#3D405B"
    ]

    color_map = {}
    europe = sorted([c for c in countries if country_key(c) in europe_country_keys])
    non_europe = [c for c in countries if c not in europe]

    for i, country in enumerate(europe):
        color_map[country] = greens[i % len(greens)]

    fallback_idx = 0
    for country in non_europe:
        key = country_key(country)
        if key in forced_colors:
            color_map[country] = forced_colors[key]
        else:
            color_map[country] = fallback_palette[fallback_idx % len(fallback_palette)]
            fallback_idx += 1

    return color_map


def format_hover(row: pd.Series) -> str:
    lines = [f"<b>{row['actor']}</b>"]

    if pd.notna(row.get("sector")):
        lines.append(f"Sector: {row['sector']}")
    if pd.notna(row.get("country")):
        lines.append(f"Country: {row['country']}")
    if pd.notna(row.get("founded_year")):
        lines.append(f"Founded: {int(row['founded_year'])}")
    if pd.notna(row.get("employees_count")):
        lines.append(f"Employees: {int(row['employees_count']):,}")

    cap = row.get("capitalization_num")
    if pd.notna(cap) and cap > 0:
        lines.append(f"Market cap: {cap/1000:.1f}B USD")

    rev = row.get("revenue_num")
    if pd.notna(rev) and rev > 0:
        lines.append(f"Revenue: {rev/1000:.1f}B USD")

    if pd.notna(row.get("description")):
        desc = str(row["description"])
        snippet = desc[:160].rstrip()
        lines.append(f"<i>{snippet}{'…' if len(desc) > 160 else ''}</i>")

    lines.append(f"Outgoing competitor links: {int(row['score'])}")
    return "<br>".join(lines)


# Assemble les donnees de plotting.
df_plot = df_coords.copy()
available = [c for c in info_cols if c in df_raw.columns]
if available:
    df_info = df_raw.set_index(COL_NAME)[available]
    for col in available:
        df_plot[col] = df_plot["actor"].map(df_info[col])

# Ajoute le pays avec fallback SQL si la colonne manque dans df_raw.
if "country" in df_raw.columns:
    country_map = df_raw.set_index(COL_NAME)["country"]
else:
    with sqlite3.connect(DB_PATH) as con:
        df_country = pd.read_sql_query(f"SELECT name AS {COL_NAME}, country FROM enterprises", con)
    country_map = df_country.set_index(COL_NAME)["country"]

df_plot["country"] = df_plot["actor"].map(country_map).fillna("Unknown")

# Colonnes numeriques utiles pour hover/tailles.
df_plot["capitalization_num"] = pd.to_numeric(df_plot["capitalization"], errors="coerce") if "capitalization" in df_plot.columns else np.nan
df_plot["revenue_num"] = pd.to_numeric(df_plot["revenue_millions"], errors="coerce") if "revenue_millions" in df_plot.columns else np.nan

# Taille des labels: mode configurable, defaut log1p + clipping P99.
df_plot["label_size"], size_meta = compute_label_sizes(
    raw_scores=df_plot["ranking_score"],
    mode=SIZE_MODE,
    min_size=SIZE_MIN,
    max_size=SIZE_MAX,
    fallback_size=SIZE_FALLBACK,
    clip_q=SIZE_CLIP_Q,
)

# Repositionnement repel en deux passes: global puis micro-ajustement intra-pays.
label_xy = apply_label_repel(
    df_plot,
    x_col="x",
    y_col="y",
    size_col="label_size",
    iterations=380,
    anchor_pull=0.018,
    step=0.62,
)
df_plot["x_label"] = label_xy["x_label"]
df_plot["y_label"] = label_xy["y_label"]

for country_name, idx in df_plot.groupby("country").groups.items():
    sub = df_plot.loc[idx, ["x_label", "y_label", "label_size"]].rename(
        columns={"x_label": "x", "y_label": "y"}
    )
    sub_xy = apply_label_repel(
        sub,
        x_col="x",
        y_col="y",
        size_col="label_size",
        iterations=160,
        anchor_pull=0.045,
        step=0.36,
    )
    df_plot.loc[idx, "x_label"] = sub_xy["x_label"].to_numpy()
    df_plot.loc[idx, "y_label"] = sub_xy["y_label"].to_numpy()

mean_shift = np.mean(np.sqrt((df_plot["x_label"] - df_plot["x"]) ** 2 + (df_plot["y_label"] - df_plot["y"]) ** 2))
print(f"Repel shift moyen: {mean_shift:.2f} (unités de projection)")
print(f"Label size stats -> min: {df_plot['label_size'].min():.1f}, max: {df_plot['label_size'].max():.1f}")
print(f"Label size basis -> {size_meta['mode']} on ranking_score")
if size_meta["clip_value"] is not None:
    print(f"Clipping upper bound (P99): {size_meta['clip_value']:.3g}")

# Hover + couleurs par pays.
df_plot["hover"] = df_plot.apply(format_hover, axis=1)
countries = sorted(df_plot["country"].dropna().unique().tolist())
country_colors = build_country_colors(countries)

fig = go.Figure()
for country in countries:
    sub = df_plot[df_plot["country"] == country].copy().sort_values("label_size", ascending=True)

    sub["label_text"] = sub["actor"].combine(sub["label_size"], lambda actor, size: (
        f"<span style='font-size:{size}px;font-weight:400;"
        "text-shadow:0 0 1px rgba(253,250,244,0.95),0 0 3px rgba(253,250,244,0.65)'>"
        f"{actor}</span>"
    ))

    fig.add_trace(
        go.Scatter(
            x=sub["x_label"],
            y=sub["y_label"],
            mode="text",
            text=sub["label_text"],
            textposition="middle center",
            textfont=dict(color=country_colors[country]),
            name=country,
            customdata=np.stack([sub["hover"]], axis=-1),
            hovertemplate="%{customdata[0]}<extra></extra>",
            showlegend=True,
        )
    )

fig.update_layout(
    title=f"{method} — Entreprises dans l'espace des competiteurs ({N} entreprises)",
    xaxis=dict(title="Dimension 1", showgrid=False, zeroline=False),
    yaxis=dict(title="Dimension 2", showgrid=False, zeroline=False),
    legend=dict(title="Pays", font=dict(size=11)),
    font=dict(family="Inter, sans-serif", size=12),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
    width=1400,
    height=1200,
)

fig_html = EXPORTS_DIR / "competition_map_2d.html"
fig.write_html(str(fig_html))
print(f"Carte interactive ({method}) -> {fig_html}")
fig.show()

Repel shift moyen: 0.86 (unités de projection)
Label size stats -> min: 8.0, max: 18.0
Label size basis -> sqrt on ranking_score
Carte interactive (t-SNE) -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d.html


In [49]:
import networkx as nx

# Regroupement par graphe de proximite: k-NN sur l'espace 2D puis Louvain.
n_points = len(df_plot)
XY = df_plot[["x", "y"]].to_numpy(dtype=float)

if n_points <= 1:
    cluster_idx = np.zeros(n_points, dtype=int)
else:
    k_neighbors = int(min(8, max(2, n_points - 1)))

    # Distances pairwise en numpy pour eviter des dependances supplementaires.
    diff = XY[:, None, :] - XY[None, :, :]
    dist = np.sqrt(np.sum(diff * diff, axis=2))
    np.fill_diagonal(dist, np.inf)

    # Echelle robuste pour convertir les distances en poids de similarite.
    finite_d = dist[np.isfinite(dist)]
    sigma = float(np.median(finite_d)) if finite_d.size else 1.0
    sigma = max(sigma, 1e-6)

    G_knn = nx.Graph()
    G_knn.add_nodes_from(range(n_points))

    for i in range(n_points):
        nn_idx = np.argsort(dist[i])[:k_neighbors]
        for j in nn_idx:
            w = float(np.exp(-dist[i, j] / sigma))
            if G_knn.has_edge(i, int(j)):
                if w > G_knn[i][int(j)]["weight"]:
                    G_knn[i][int(j)]["weight"] = w
            else:
                G_knn.add_edge(i, int(j), weight=w)

    communities = nx.community.louvain_communities(
        G_knn,
        weight="weight",
        seed=RANDOM_SEED,
        resolution=1.0,
    )

    # Trie par taille decroissante pour stabiliser les IDs de groupe.
    communities = sorted(communities, key=len, reverse=True)

    # On garde les 6 principaux groupes comme precedemment.
    top_k = 6
    cluster_idx = np.full(n_points, -1, dtype=int)
    for cid, members in enumerate(communities[:top_k]):
        for node in members:
            cluster_idx[node] = cid

    # Les points hors top_k sont rattaches au groupe voisin le plus proche.
    if np.any(cluster_idx < 0):
        assigned = np.where(cluster_idx >= 0)[0]
        unassigned = np.where(cluster_idx < 0)[0]
        if len(assigned) > 0:
            for u in unassigned:
                nearest = assigned[np.argmin(dist[u, assigned])]
                cluster_idx[u] = cluster_idx[nearest]
        else:
            cluster_idx[:] = 0

df_plot["community"] = cluster_idx.astype(int)
community_method = "graph_knn_louvain_2d"

# Labels metier injectes dans la carte.
group_name_map = {
    0: "Modeles IA applicatifs",
    1: "Geants cloud IA",
    2: "Puces IA avancees",
    3: "IA appliquee metiers",
    4: "Sante IA connectee",
}

# Fallback pour d'eventuels groupes supplementaires.
for c in sorted(np.unique(cluster_idx)):
    c = int(c)
    if c >= 0 and c not in group_name_map:
        group_name_map[c] = f"G{c}"

# Export des groupes pour audit (on ecrase le meme fichier existant).
communities_path = EXPORTS_DIR / "communities_kmeans_2d.csv"
df_plot[["actor", "community", "country", "sector", "ranking_score", "x", "y"]].sort_values(
    ["community", "ranking_score"], ascending=[True, False]
).to_csv(communities_path, index=False)

palette = [
    "#1B4332", "#2D6A4F", "#40916C", "#74C69D", "#0A9396", "#005F73",
    "#CA6702", "#BB3E03", "#AE2012", "#6D597A", "#355070", "#588157",
]

community_values = sorted(c for c in df_plot["community"].unique() if c >= 0)
community_colors = {c: palette[i % len(palette)] for i, c in enumerate(community_values)}
community_sizes = df_plot.groupby("community").size().sort_values(ascending=False)
main_communities = [int(c) for c in community_sizes.index if c >= 0][:6]


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def convex_hull(points: np.ndarray) -> np.ndarray:
    """Monotonic chain convex hull. Returns vertices in order."""
    pts = np.unique(points, axis=0)
    if len(pts) <= 2:
        return pts

    pts = pts[np.lexsort((pts[:, 1], pts[:, 0]))]

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(tuple(p))

    upper = []
    for p in pts[::-1]:
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(tuple(p))

    hull = np.array(lower[:-1] + upper[:-1], dtype=float)
    return hull


def chaikin_smooth(closed_poly: np.ndarray, refinements: int = 2) -> np.ndarray:
    """Smooth a closed polygon using Chaikin corner cutting."""
    poly = closed_poly.copy()
    for _ in range(refinements):
        new_pts = []
        for i in range(len(poly) - 1):
            p0 = poly[i]
            p1 = poly[i + 1]
            q = 0.75 * p0 + 0.25 * p1
            r = 0.25 * p0 + 0.75 * p1
            new_pts.extend([q, r])
        poly = np.vstack([new_pts, new_pts[0]])
    return poly


def build_group_patch(sub: pd.DataFrame, pad: float) -> tuple[np.ndarray, np.ndarray, float, float]:
    pts = sub[["x_label", "y_label"]].to_numpy(dtype=float)
    cx = float(np.mean(pts[:, 0]))
    cy = float(np.mean(pts[:, 1]))

    if len(pts) >= 3:
        hull = convex_hull(pts)
        if len(hull) >= 3:
            vec = hull - np.array([cx, cy])
            dist_local = np.linalg.norm(vec, axis=1, keepdims=True)
            scale = (dist_local + pad) / np.maximum(dist_local, 1e-6)
            inflated = np.array([cx, cy]) + vec * scale
            closed = np.vstack([inflated, inflated[0]])
            smooth = chaikin_smooth(closed, refinements=2)
            return smooth[:, 0], smooth[:, 1], cx, cy

    # Fallback shape for tiny groups.
    r = max(0.7, pad)
    theta = np.linspace(0, 2 * np.pi, 90)
    x_blob = cx + (r * 1.15) * np.cos(theta)
    y_blob = cy + (r * 0.90) * np.sin(theta)
    return x_blob, y_blob, cx, cy


fig_louvain = go.Figure()
community_centroids = {}

# Patates: enveloppe convexe lissee, plus fidele aux points que le blob radial.
for c in main_communities:
    sub = df_plot[df_plot["community"] == c]
    if len(sub) < 2:
        continue

    pad = float(max(0.8, sub["label_size"].mean() * 0.24))
    x_blob, y_blob, cx, cy = build_group_patch(sub, pad=pad)

    community_centroids[c] = (cx, cy, int(len(sub)))
    color = community_colors[c]
    label_name = group_name_map.get(int(c), f"G{c}")

    fig_louvain.add_trace(
        go.Scatter(
            x=x_blob,
            y=y_blob,
            mode="lines",
            fill="toself",
            fillcolor=hex_to_rgba(color, 0.12),
            line=dict(color=hex_to_rgba(color, 0.55), width=2.0),
            name=f"Zone {label_name}",
            hovertemplate=f"{label_name}<br>Groupe {c}<br>Acteurs: {len(sub)}<extra></extra>",
            showlegend=False,
        )
    )

# Labels colores par pays, avec groupe dans le hover.
if "country_colors" not in globals():
    countries = sorted(df_plot["country"].dropna().unique().tolist())
    country_colors = build_country_colors(countries)

countries = sorted(df_plot["country"].dropna().unique().tolist())
for country in countries:
    sub = df_plot[df_plot["country"] == country].copy().sort_values("label_size", ascending=True)

    sub["label_text"] = sub["actor"].combine(sub["label_size"], lambda actor, size: (
        f"<span style='font-size:{size}px;font-weight:500;'"
        "text-shadow:0 0 1px rgba(253,250,244,0.95),0 0 3px rgba(253,250,244,0.65)'>"
        f"{actor}</span>"
    ))

    fig_louvain.add_trace(
        go.Scatter(
            x=sub["x_label"],
            y=sub["y_label"],
            mode="text",
            text=sub["label_text"],
            textposition="middle center",
            textfont=dict(color=country_colors.get(country, "#374151")),
            name=country,
            customdata=np.stack([sub["hover"], sub["community"]], axis=-1),
            hovertemplate="%{customdata[0]}<br>Groupe: %{customdata[1]}<extra></extra>",
            showlegend=True,
        )
    )

community_annotations = []
for c, (cx, cy, n_members) in community_centroids.items():
    label_name = group_name_map.get(int(c), f"G{c}")
    community_annotations.append(
        dict(
            x=cx,
            y=cy,
            text=f"{label_name}<br>(n={n_members})",
            showarrow=False,
            font=dict(size=11, color="#111827"),
            bgcolor="rgba(253,250,244,0.88)",
            bordercolor=hex_to_rgba(community_colors[c], 0.75),
            borderwidth=1,
            borderpad=3,
        )
    )

fig_louvain.update_layout(
    title=f"{method} — Groupes de proximite 2D ({community_method})",
    xaxis=dict(title="Dimension 1", showgrid=False, zeroline=False),
    yaxis=dict(title="Dimension 2", showgrid=False, zeroline=False),
    legend=dict(title="Pays", font=dict(size=11)),
    font=dict(family="Inter, sans-serif", size=12),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
    width=1400,
    height=1200,
    annotations=community_annotations,
)

# On ecrase la meme sortie HTML existante.
fig_louvain_html = EXPORTS_DIR / "competition_map_2d_kmeans.html"
fig_louvain.write_html(str(fig_louvain_html))

print(f"Groupes detectes: {len(community_values)} ({community_method})")
print(f"Groupes principaux entoures et labelises: {len(main_communities)}")
print("Labels metier: " + " | ".join([f"G{k}={v}" for k, v in sorted(group_name_map.items())]))
print(f"Export groupes -> {communities_path}")
print(f"Carte (remplacement) -> {fig_louvain_html}")
fig_louvain.show()

Groupes detectes: 5 (graph_knn_louvain_2d)
Groupes principaux entoures et labelises: 5
Labels metier: G0=Modeles IA applicatifs | G1=Geants cloud IA | G2=Puces IA avancees | G3=IA appliquee metiers | G4=Sante IA connectee
Export groupes -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\communities_kmeans_2d.csv
Carte (remplacement) -> C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d_kmeans.html


## 8b. Regroupement Graphe (k-NN + Louvain) dans l'espace 2D

**Objectif**: regrouper les entreprises par voisinages locaux dans la projection 2D.

**Rendu**:
- Labels gardes en couleur pays (comme la carte principale).
- Patates basees sur l'enveloppe convexe lisse de chaque groupe.
- Labels de groupe au centre des zones.

**Methode**:
- Construction d'un graphe k-NN sur les coordonnees 2D (`x`, `y`).
- Poids de similarite bases sur la distance euclidienne.
- Detection de communautes Louvain sur ce graphe.
- Conservation des 6 groupes principaux (les autres points sont rattaches au groupe le plus proche).

**Labels metier injectes**:
- G0: `Modeles IA applicatifs`
- G1: `Geants cloud IA`
- G2: `Puces IA avancees`
- G3: `IA appliquee metiers`
- G4: `Sante IA connectee`

**Sorties**:
- `communities_kmeans_2d.csv` (fichier remplace)
- `competition_map_2d_kmeans.html` (fichier remplace)

In [50]:
print("── Récapitulatif des exports ────────────────────────────")
export_paths = [raw_path, long_path, agg_path, cooc_path, coords_path, fig_html]

for maybe_var in ["communities_path", "fig_louvain_html"]:
    if maybe_var in globals():
        export_paths.append(globals()[maybe_var])

# Ajoute explicitement les sorties KMeans si presentes.
for p in [
    EXPORTS_DIR / "communities_kmeans_2d.csv",
    EXPORTS_DIR / "competition_map_2d_kmeans.html",
]:
    if p not in export_paths:
        export_paths.append(p)

# Supprime les doublons en conservant l'ordre.
seen = set()
ordered_paths = []
for p in export_paths:
    p = Path(p)
    key = str(p)
    if key not in seen:
        seen.add(key)
        ordered_paths.append(p)

for p in ordered_paths:
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:<42} {size_kb:6.1f} KB")
    else:
        print(f"  {p.name:<42} {'MISSING':>6}")

── Récapitulatif des exports ────────────────────────────
  competitors_raw.csv                          11.0 KB
  competitors_long.csv                         32.4 KB
  competitors_aggregated.csv                   12.4 KB
  cooccurrence_matrix.csv                     140.2 KB
  coords_2d.csv                                 6.8 KB
  competition_map_2d.html                    4784.8 KB
  communities_kmeans_2d.csv                     5.9 KB
  competition_map_2d_kmeans.html             4793.4 KB
